# Notebook 01 — Data Pipeline & Exploration
**Syllabus Week 5 | CLO 2: Data Preparation and Feature Engineering**

## Problem Statement
Build a clean, reproducible weather dataset for 6 Pakistani cities covering 2023–2024.
The dataset must support:
- Time-series forecasting (Transformer, LSTM)
- Point-in-time anomaly detection (VAE)
- Visual cross-referencing (CNN / CLIP)

**Source:** Open-Meteo Archive API (`/v1/archive`) — hourly, 2023-01-01 → 2024-12-31.

**Cities:** Karachi · Lahore · Islamabad · Topi · Peshawar · Quetta

**Features (7):** `temperature_2m`, `relative_humidity_2m`, `precipitation`,
`wind_speed_10m`, `weather_code`, `surface_pressure`, `cloud_cover`

**Time split (NEVER random-split):**
| Split | Range | Rows/city |
|---|---|---|
| Train | 2023-01-01 – 2023-12-31 | ~8,760 |
| Val   | 2024-01-01 – 2024-06-30 | ~4,368 |
| Test  | 2024-07-01 – 2024-12-31 | ~4,416 |

## Section 2 — Key Operations & Equations

### Z-score Standardisation (train-only)
For feature $x$ with train-split mean $\mu$ and std $\sigma$:
$$\hat{x} = \frac{x - \mu}{\sigma}$$
Scaler fit on **train rows only** to prevent val/test leakage.

### Forward-Fill Imputation
For missing value at position $t$, forward-fill from position $t-k$ where $k \le 3$:
$$x_t \leftarrow x_{t-k}, \quad k = \min\{j \ge 1 : x_{t-j} \text{ is not NaN}\}$$
If gap exceeds 3 hours, NaN is retained and logged to `imputation_log.json`.

### Sliding Window
For a time series of length $T$, input length $L=48$, horizon $H=24$:
$$\text{Number of windows} = T - L - H + 1$$
Each window: $X_i = [x_{i}, \ldots, x_{i+L-1}] \in \mathbb{R}^{L \times F}$,
$y_i = [x^{\text{temp}}_{i+L}, \ldots, x^{\text{temp}}_{i+L+H-1}] \in \mathbb{R}^H$

In [ ]:
# Section 3 — Data Loading & Preprocessing
# TODO: fill in after running python -m ml.data_pipeline
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATASETS = Path('../datasets')
CITIES = ['karachi','lahore','islamabad','topi','peshawar','quetta']

# TODO: load parquets and report shapes
# dfs = {city: pd.read_parquet(DATASETS / f'{city}.parquet') for city in CITIES}
# for city, df in dfs.items():
#     print(f'{city}: {df.shape}  NaN: {df.isna().sum().sum()}')  # (rows, 8)

# TODO: show split row counts
# train = df.loc[df.index <= '2023-12-31']   # shape: (~8760, 7)
# val   = df.loc['2024-01-01':'2024-06-30']   # shape: (~4368, 7)
# test  = df.loc['2024-07-01':]               # shape: (~4416, 7)

In [ ]:
# Section 4 — Scaler Stats & Window Construction
# TODO: load scaler_stats.json and verify no test-split contamination

# with open(DATASETS / 'scaler_stats.json') as f:
#     stats = json.load(f)
# print('Features in scaler:', list(stats.keys()))
# for feat, s in stats.items():
#     print(f'  {feat:30s}  mean={s["mean"]:8.3f}  std={s["std"]:7.3f}')

# from ml.data_pipeline import make_windows, apply_scaler
# df_scaled = apply_scaler(df, stats)
# X, y = make_windows(df_scaled, input_len=48, horizon=24)
# print(f'X shape: {X.shape}')  # (N, 48, 7)
# print(f'y shape: {y.shape}')  # (N, 24)

In [ ]:
# Section 5 — Exploratory Analysis
# TODO: run after bootstrap
import matplotlib.pyplot as plt

# TODO: plot temperature time series for all cities
# fig, axes = plt.subplots(6, 1, figsize=(14, 18), sharex=True)
# for ax, (city, df) in zip(axes, dfs.items()):
#     ax.plot(df['temperature_2m'], lw=0.4, label=city)
#     ax.set_ylabel('Temp (C)'); ax.legend(loc='upper right')
# axes[-1].set_xlabel('Date')
# plt.suptitle('Temperature 2023-2024, 6 PK Cities')
# plt.tight_layout(); plt.show()

# TODO: correlation heatmap of features (Lahore train split)
# import seaborn as sns
# sns.heatmap(train[FEATURES].corr(), annot=True, fmt='.2f', cmap='coolwarm')
# plt.title('Feature Correlation — Lahore 2023')
# plt.tight_layout(); plt.show()

## Section 6 — Pipeline Architecture

```mermaid
graph LR
    A[Open-Meteo Archive API] -->|httpx async| B[fetch_city]
    B --> C[clean: ffill le 3h]
    C --> D[city.parquet]
    D --> E[compute_scaler_stats\ntrain split only]
    E --> F[scaler_stats.json]
    D --> G[apply_scaler]
    G --> H[make_windows\ninput=48 horizon=24]
    H --> I[TensorDataset\nX: N x 48 x 7\ny: N x 24]
```

**Key invariant:** `scaler_stats.json` is computed from 2023 rows only.
Val and test rows are standardised using the *same* train-derived mean/std —
never refit on val or test.

In [ ]:
# Section 7 — Evaluation & Plots
# TODO: run after bootstrap

# TODO: NaN rates table
# with open(DATASETS / 'imputation_log.json') as f:
#     log = json.load(f)
# if log:
#     print('Features with >1% NaN after imputation:')
#     for key, v in log.items():
#         print(f'  {key}: {v["nan_rate_pct"]:.2f}% ({v["post_fill_nans"]} rows)')
# else:
#     print('All features <1% NaN after imputation — clean dataset.')

# TODO: window count per city per split
# for city in CITIES:
#     df = pd.read_parquet(DATASETS / f'{city}.parquet')
#     df = apply_scaler(df, stats)
#     for split, df_s in [('train', df.loc[df.index<='2023-12-31']),
#                         ('val',   df.loc['2024-01-01':'2024-06-30']),
#                         ('test',  df.loc['2024-07-01':])]:  
#         X, y = make_windows(df_s)
#         print(f'{city}/{split}: {len(X)} windows')

## Section 8 — Reflection & Trade-offs

### Design decisions
| Decision | Alternative considered | Reason chosen |
|---|---|---|
| Forward-fill ≤ 3h only | Drop rows with any NaN | Preserves continuity; 3h is meteorologically reasonable (patterns change slowly) |
| Archive API not Forecast API | Forecast API | Forecast only returns ~92 days back; Archive covers full 2-year window |
| Train-only scaler | Fit on full dataset | Prevents information leakage from future val/test distributions |
| Chronological split | Random split | Time-series models must never see future data during training |
| Parquet format | CSV | ~10× smaller, preserves dtypes, faster I/O for repeated training runs |

### Limitations
- Only 7 features — ignores snow_depth, soil_temperature available from Open-Meteo.
- Single-point city coordinates — no spatial interpolation across city boundaries.
- `weather_code` is categorical but treated as continuous — a learned embedding would be cleaner.

### Numbers to fill in after bootstrap
- Total rows across 6 cities: **TODO**
- Max NaN rate any feature/city: **TODO**
- Total training windows: **TODO**